In [1]:
from pathlib import Path
import pandas as pd
import sys
from catboost import CatBoostClassifier, Pool, EFstrType
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

from src.utils.preprocess import (
    preprocess_initial_dataset,
    split_dataset,
    remove_log_features,
    split_dataset_with_val,
    train_catboost_model,
    get_feature_importance_df
)


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

DATA_PATH = Path("../../data/cleaned_dataset/final_dataset_from_notebooks.csv")
RAW_DATA_PATH = Path("../../data/raw/clean_dataset.csv")

In [2]:
df = pd.read_csv(DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)
df = df.merge(
    df_raw[
        ["row_id", "lead_Стоимость доставки", "lead_Масса (гр)", "lead_Высота"]
    ],
    on="row_id",
    how="left",
)
df = remove_log_features(df)
del df_raw
# Приводим всё к нужному типу
df = preprocess_initial_dataset(df)
df.info()

C:\Users\Михаил\AppData\Local\Temp\ipykernel_18504\3613829215.py:2: DtypeWarning: Columns (79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(RAW_DATA_PATH)


<class 'pandas.core.frame.DataFrame'>
Index: 17966 entries, 0 to 18787
Columns: 110 entries, contact_pvz_code to contact_has_pvz_code
dtypes: datetime64[ns](2), float64(18), int64(64), object(26)
memory usage: 15.2+ MB


In [3]:
print(*df.columns.tolist(), sep="\n")

contact_pvz_code
lead_weight_gm
lead_responsible_user_id
lead_utm_group
lead_utm_referrer
lead_tags
lead_utm_source
lead_Квалификация лида
is_responsible_same
has_weight
contact_region_pvz
lead_utm_referrer_site
lead_has_roistat
lead_utm_id_1
lead_utm_id_2
lead_utm_id_3
lead_utm_device_type
lead_utm_site
lead_utm_position
lead_utm_reatrgeting_id
lead_utm_region_name
lead_is_utm_campaign_type_1
contact_Город
contact_region
lead_manager_category
lead_rate_is_warehouse_to_warehouse
lead_formname_has_value
lead_has_creation_date
lead_creation_date_week
lead_creation_date_month
lead_creation_date_quarter
lead_creation_date_dayofweek
lead_creation_date_sin
lead_creation_date_cos
sale_date_month
sale_date_quarter
sale_date_dayofweek
sale_date_week
sale_date_sin
sale_date_cos
lead_items_count
lead_total_quantity
lead_total_cost_from_composition
lead_has_health_supplement
lead_has_pillow
lead_has_mattress
lead_has_brace
lead_has_footwear
lead_has_accessory
lead_has_health_product
lead_categorie

Удаляем из модели субъективную оценку менеджеров

Удаляем lead_group_quality так как потенциально содержит данные из будущего

In [ ]:
df_reduced = df.drop(columns=["lead_Квалификация лида", "lead_group_quality"])

X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_reduced)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления lead_group_quality:", metrics)

0:	learn: 0.6245627	test: 0.6546443	best: 0.6546443 (0)	total: 233ms	remaining: 7m 45s
100:	learn: 0.7758491	test: 0.6736037	best: 0.6893284 (27)	total: 7.23s	remaining: 2m 15s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6893284204
bestIteration = 27

Shrink model to first 28 iterations.
Метрики после удаления lead_group_quality: {'ROC_AUC': 0.671643587085308, 'PR_AUC': 0.3917883822509819, 'LogLoss': 0.8087788282483225}


In [4]:
fi = get_feature_importance_df(model, X_train, y_train)

fi.head(20)

,feature,importance
71,lead_payment_type,20.269806
72,lead_delivery_type,7.227213
5,lead_tags,7.037723
3,lead_utm_group,5.106532
15,lead_utm_device_type,4.976128
14,lead_utm_id_3,4.417822
57,lead_source,3.770477
59,lead_created_ts,2.869306
99,lead_height_known,2.677279
10,lead_utm_referrer_site,2.608297


In [6]:
baseline_pr_auc = 0.3917883822509819

results = []

for feature in fi["feature"]:
    print(f"\nУдаляем признак: {feature}")

    df_removed = df_reduced.drop(columns=[feature])

    X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
        df_removed
    )

    model, metrics = train_catboost_model(
        X_train, y_train, X_val, y_val, X_test, y_test
    )

    pr_auc = metrics["PR_AUC"]

    print(f"PR_AUC: {pr_auc:.6f}")

    results.append((feature, pr_auc))

    # проверка улучшения
    if pr_auc > baseline_pr_auc:
        print(f"✅ Улучшение! Удаление признака {feature} дало PR_AUC = {pr_auc:.6f}")


Удаляем признак: lead_payment_type
0:	learn: 0.6530347	test: 0.6248851	best: 0.6248851 (0)	total: 72.7ms	remaining: 2m 25s
100:	learn: 0.7784653	test: 0.6705798	best: 0.6814501 (32)	total: 7.21s	remaining: 2m 15s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6814500714
bestIteration = 32

Shrink model to first 33 iterations.
PR_AUC: 0.358004

Удаляем признак: lead_delivery_type
0:	learn: 0.6552539	test: 0.6663079	best: 0.6663079 (0)	total: 59.2ms	remaining: 1m 58s
100:	learn: 0.7685274	test: 0.6631908	best: 0.6833769 (5)	total: 6.78s	remaining: 2m 7s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.683376936
bestIteration = 5

Shrink model to first 6 iterations.
PR_AUC: 0.362281

Удаляем признак: lead_tags
0:	learn: 0.6542625	test: 0.6392871	best: 0.6392871 (0)	total: 64.9ms	remaining: 2m 9s
100:	learn: 0.7794893	test: 0.6539005	best: 0.6640401 (26)	total: 7.01s	remaining: 2m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTes

In [7]:
results_df = pd.DataFrame(results, columns=["feature", "pr_auc"])
results_df = results_df.sort_values(by="pr_auc", ascending=False)

print(results_df[results_df["pr_auc"] > baseline_pr_auc])

Empty DataFrame
Columns: [feature, pr_auc]
Index: []


In [8]:
baseline_pr_auc = 0.3917883822509819

for n in range(90, 29, -1):
    col_to_drop = fi["feature"].tail(n).to_list()

    df_test = df_reduced.drop(columns=col_to_drop)

    X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(df_test)

    model, metrics = train_catboost_model(
        X_train, y_train, X_val, y_val, X_test, y_test
    )

    pr_auc = metrics["PR_AUC"]

    print(f"tail({n}) -> PR_AUC: {pr_auc:.6f}")

    if pr_auc > baseline_pr_auc:
        print("\nIMPROVEMENT FOUND")
        print(f"tail({n}) gives PR_AUC = {pr_auc:.6f} (baseline = {baseline_pr_auc})")
        break

0:	learn: 0.6504361	test: 0.6674674	best: 0.6674674 (0)	total: 43.4ms	remaining: 1m 26s
100:	learn: 0.7324285	test: 0.6802772	best: 0.6841077 (89)	total: 4.19s	remaining: 1m 18s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6841076967
bestIteration = 89

Shrink model to first 90 iterations.
tail(90) -> PR_AUC: 0.357177
0:	learn: 0.6373416	test: 0.6115798	best: 0.6115798 (0)	total: 38.7ms	remaining: 1m 17s
100:	learn: 0.7287504	test: 0.6777657	best: 0.6782845 (97)	total: 4.02s	remaining: 1m 15s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6782844836
bestIteration = 97

Shrink model to first 98 iterations.
tail(89) -> PR_AUC: 0.353229
0:	learn: 0.6605575	test: 0.6317222	best: 0.6317222 (0)	total: 40.8ms	remaining: 1m 21s
100:	learn: 0.7265317	test: 0.6707890	best: 0.6809839 (57)	total: 4.15s	remaining: 1m 17s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6809839358
bestIteration = 57

Shrink model to first 58 iteration

In [9]:
col_to_drop = fi["feature"].tail(61).to_list()

df_reduced = df_reduced.drop(columns=col_to_drop)
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset_with_val(
    df_reduced
)

model, metrics = train_catboost_model(
    X_train, y_train, X_val, y_val, X_test, y_test
)

print("Метрики после удаления признаков от sale_ts:", metrics)

0:	learn: 0.6546364	test: 0.6738810	best: 0.6738810 (0)	total: 50.4ms	remaining: 1m 40s
100:	learn: 0.7571080	test: 0.6652833	best: 0.6885438 (11)	total: 5.19s	remaining: 1m 37s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6885438085
bestIteration = 11

Shrink model to first 12 iterations.
Метрики после удаления признаков от sale_ts: {'ROC_AUC': 0.6820383907413676, 'PR_AUC': 0.3981714836743723, 'LogLoss': 0.7643426028525783}


In [12]:
X_train.columns.to_list()

['lead_responsible_user_id',
 'lead_utm_group',
 'lead_tags',
 'contact_region_pvz',
 'lead_utm_referrer_site',
 'lead_utm_id_1',
 'lead_utm_id_2',
 'lead_utm_id_3',
 'lead_utm_device_type',
 'lead_utm_reatrgeting_id',
 'lead_manager_category',
 'lead_creation_date_month',
 'lead_creation_date_quarter',
 'lead_creation_date_sin',
 'sale_date_quarter',
 'sale_date_dayofweek',
 'lead_total_cost_from_composition',
 'lead_has_brace',
 'lead_discount_category',
 'sale_hour',
 'sale_month',
 'lead_source',
 'timedelta_between_sale_and_creation',
 'lead_created_ts',
 'lead_created_dayofweek',
 'lead_shipping_cost',
 'lead_length',
 'lead_price',
 'lead_group_id',
 'width_cat',
 'width_is_missing',
 'lead_payment_type',
 'lead_delivery_type',
 'contact_to_lead_hours',
 'contact_hour',
 'contact_month',
 'utm_sky_autotarget',
 'utm_sky_brand',
 'lead_group_grouped',
 'lead_height_known',
 'lead_height_bin',
 'delivery_cost_missing',
 'lead_Стоимость доставки',
 'lead_Масса (гр)',
 'lead_Высота'